# 3. Identity Governance

Identity governance answers the question: **"Do the right people have the right access to the right resources at the right time?"**

This is about ongoing management — not just granting access, but reviewing it, limiting it over time, and detecting anomalies.

## What Microsoft Entra ID Governance covers

Microsoft frames it as three lifecycles — the **identity** lifecycle (joiner/mover/leaver),
the **access** lifecycle (who may request what, and for how long), and the **privileged access**
lifecycle (admin rights). The features that implement them:

| Capability | What it does | When to use it |
|------------|-------------|----------------|
| **Entitlement Management** | Bundles of resources ("access packages") that users can request | Onboarding, project teams, partner access |
| **Access Reviews** | Periodic certification that access is still needed | Quarterly reviews, compliance audits |
| **Privileged Identity Management (PIM)** | Just-in-time elevation for admin roles | Anyone with admin access |
| **Lifecycle Workflows** | Automates joiner/mover/leaver tasks from an HR signal | Day-one provisioning, offboarding |

> ⚠️ **Microsoft Entra ID Protection** gets a section in this notebook because the exam objective
> reads *"identity protection **and** governance capabilities"* — but it is not part of the
> Entra ID Governance product. ID Protection is a **risk-detection** service: it scores sign-ins
> and users, and feeds those scores to Conditional Access. Governance decides *who should have
> access*; ID Protection decides *whether this particular sign-in smells wrong*.

---
## 1. Privileged Identity Management (PIM)

**Problem**: people get admin roles and keep them forever. A compromised admin account = total breach.

**PIM solution**: admin roles are **eligible**, not **active**. When someone needs admin access, they **activate** the role for a limited time (e.g., 4 hours), optionally requiring approval and MFA.

### How PIM works

```
Alice (eligible for Global Admin)
  │
  ├── Normal day: no admin permissions (Reader only)
  │
  └── Needs to change a policy:
        1. Opens PIM portal
        2. Requests "Global Admin" activation
        3. Provides justification: "Need to update CA policy for APAC"
        4. Completes MFA challenge
        5. Manager approves (if approval required)
        6. Role active for 4 hours
        7. Automatically deactivates
```

### Key PIM concepts for the exam

| Concept | Meaning |
|---------|----------|
| **Eligible assignment** | User *can* activate the role but doesn't have it by default |
| **Active assignment** | User has the role right now (permanent or time-limited) |
| **Activation** | The act of temporarily enabling an eligible role |
| **Justification** | Required text explaining *why* the role is needed |
| **Approval workflow** | Another person must approve the activation |
| **Maximum activation duration** | How long the role stays active (e.g., 8 hours max) |

**Exam tip**: PIM requires **Entra ID Premium P2** (or Entra ID Governance license).

In [ ]:
import json
from datetime import datetime, timedelta

# Simulate PIM role assignments
PIM_ASSIGNMENTS = {
    'alice': {'role': 'Global Administrator', 'type': 'eligible', 'max_duration_hours': 4},
    'bob':   {'role': 'User Administrator',   'type': 'eligible', 'max_duration_hours': 8},
    'carol': {'role': 'Security Reader',      'type': 'active',   'permanent': True},
}

def activate_role(user: str, justification: str, mfa_done: bool, approved: bool) -> dict:
    assignment = PIM_ASSIGNMENTS.get(user)
    if not assignment:
        return {'result': '❌ No PIM assignment found'}
    if assignment['type'] == 'active':
        return {'result': f'ℹ️ {user} already has active "{assignment["role"]}" role'}
    
    checks = []
    checks.append(('Justification provided', bool(justification)))
    checks.append(('MFA completed', mfa_done))
    checks.append(('Approval granted', approved))
    
    all_passed = all(c[1] for c in checks)
    now = datetime.now()
    return {
        'user': user,
        'role': assignment['role'],
        'checks': [{'check': c[0], 'result': '✅' if c[1] else '❌'} for c in checks],
        'result': f'✅ ACTIVATED until {(now + timedelta(hours=assignment["max_duration_hours"])).strftime("%H:%M")}' if all_passed else '❌ DENIED',
        'justification': justification,
    }

print('=== Scenario 1: Proper activation ===')
print(json.dumps(activate_role('alice', 'Need to update CA policy for APAC rollout', True, True), indent=2))

print('\n=== Scenario 2: No MFA ===')
print(json.dumps(activate_role('alice', 'Need to update CA policy', False, True), indent=2))

print('\n=== Scenario 3: No justification ===')
print(json.dumps(activate_role('bob', '', True, True), indent=2))

print('\n=== Scenario 4: Already active ===')
print(json.dumps(activate_role('carol', 'n/a', True, True), indent=2))

# The point of PIM is that activation is *conditional*. Assert each gate really gates.
assert activate_role('alice', 'valid reason', True, True)['result'].startswith('✅')
assert activate_role('alice', 'valid reason', False, True)['result'] == '❌ DENIED', \
    'activation without MFA must be denied'
assert activate_role('bob', '', True, True)['result'] == '❌ DENIED', \
    'activation without a justification must be denied'
assert activate_role('alice', 'valid reason', True, False)['result'] == '❌ DENIED', \
    'activation without approval must be denied'
assert 'already has active' in activate_role('carol', 'n/a', True, True)['result']
print('✅ every PIM gate (justification, MFA, approval) is enforced')

### Standing admin vs just-in-time admin

Below we simulate a breach of Alice's account under two setups: (1) she's a permanent Global Administrator (the old way), and (2) she's only *eligible* via PIM (the best-practice way).

In [ ]:
def attacker_uses_stolen_credentials(user_state):
    if user_state['active_admin']:
        return '💥 Full tenant compromise — attacker has Global Admin right now'
    return '🛡️  Attacker has a user session but NO admin rights until PIM activation (blocked by MFA + approval)'

print('❌ BAD: Alice is a standing Global Admin')
print('  →', attacker_uses_stolen_credentials({'active_admin': True}))

print('\n✅ GOOD: Alice is *eligible* via PIM, not active by default')
print('  →', attacker_uses_stolen_credentials({'active_admin': False}))

print('\nPIM reduces the blast radius: standing admins ≈ 0, so a stolen password is not enough.')

assert '💥' in attacker_uses_stolen_credentials({'active_admin': True}), \
    'the standing-admin case must actually show the breach, or the lab has nothing to compare'
assert '🛡️' in attacker_uses_stolen_credentials({'active_admin': False})

---
## 2. Access Reviews

**Problem**: users accumulate permissions over time ("permission creep"). Someone who changed teams 2 years ago still has access to the old team's SharePoint.

**Access reviews** are periodic checks where managers or resource owners confirm that each person's access is still needed.

| Setting | Options |
|---------|----------|
| **Reviewers** | Managers, group owners, self-review, specific users |
| **Frequency** | One-time, weekly, monthly, quarterly, annually |
| **Scope** | Group membership, app assignments, Entra/Azure roles |
| **If reviewer doesn't respond** | Remove access, approve, or no change |
| **Auto-apply results** | Automatically remove access for denied users |

### Exam tip

Access reviews can target: group members, app users, Entra role holders, and Azure role holders. They help with **compliance** (SOX, GDPR) and **least privilege**.

In [ ]:
# Simulate an access review.
# The review date is pinned rather than read from the clock: "last used 2024-08-22" has to keep
# meaning "stale" no matter when you run this notebook.
REVIEW_DATE = datetime(2026, 8, 21)

GROUP_MEMBERS = [
    {'user': 'alice@contoso.com', 'added': '2024-01-15', 'last_used': '2026-04-10', 'department': 'Engineering'},
    {'user': 'bob@contoso.com',   'added': '2023-06-01', 'last_used': '2024-08-22', 'department': 'Marketing'},
    {'user': 'carol@contoso.com', 'added': '2025-11-20', 'last_used': '2026-04-15', 'department': 'Engineering'},
    {'user': 'dave@contoso.com',  'added': '2023-01-01', 'last_used': '2023-03-15', 'department': 'Former employee'},
    {'user': 'erin@contoso.com',  'added': '2026-02-02', 'last_used': '2026-08-14', 'department': 'Marketing'},
]

ICON = {'APPROVE': '🟢', 'REVIEW': '🟡', 'DENY': '🔴'}

def recommend(member: dict) -> tuple[str, str]:
    """What a reviewer would be nudged towards. Real access reviews call these 'decision helpers'."""
    days_since_use = (REVIEW_DATE - datetime.strptime(member['last_used'], '%Y-%m-%d')).days
    if member['department'] == 'Former employee':
        return 'DENY', 'former employee, remove immediately'
    if days_since_use > 365:
        return 'DENY', f'last used {days_since_use} days ago'
    if member['department'] != 'Engineering':
        return 'REVIEW', f'department is {member["department"]}, not Engineering'
    return 'APPROVE', 'active user, correct department'

print('=== Access Review: "Engineering Repo Access" group ===')
print('Reviewer: engineering-lead@contoso.com')
print(f'Review date: {REVIEW_DATE:%Y-%m-%d}   Auto-remove if no response in 14 days\n')

decisions = {}
for member in GROUP_MEMBERS:
    verdict, reason = recommend(member)
    decisions[member['user']] = verdict
    print(f'{member["user"]}')
    print(f'  Added: {member["added"]}  Last used: {member["last_used"]}  Dept: {member["department"]}')
    print(f'  Recommendation: {ICON[verdict]} {verdict} — {reason}\n')

assert decisions == {
    'alice@contoso.com': 'APPROVE',
    'bob@contoso.com':   'DENY',      # stale: 729 days since last use
    'carol@contoso.com': 'APPROVE',
    'dave@contoso.com':  'DENY',      # left the company
    'erin@contoso.com':  'REVIEW',    # recent, but wrong department
}, f'recommendations drifted: {decisions}'
assert set(decisions.values()) == {'APPROVE', 'DENY', 'REVIEW'}, \
    'the sample must exercise all three outcomes, or a branch is dead code'
print('✅ all three review outcomes are represented')

---
## 3. Microsoft Entra ID Protection

ID Protection scores sign-ins and users against Microsoft's identity telemetry —
it does not stop anything by itself. It produces **risk**, and Conditional Access acts on it.

### Sign-in risk (is *this session* suspicious?)

| Detection | What it means |
|-----------|---------------|
| Anonymous IP address | Sign-in from Tor or an anonymising VPN |
| Atypical travel | Two sign-ins too far apart to be the same person travelling |
| Malicious IP address | Sign-in from an IP with a bad reputation |
| Unfamiliar sign-in properties | Location, device, ASN or browser never seen for this user |
| Password spray | Microsoft observed a spray attack and this user's password was successfully validated |
| Verified threat actor IP | IP tied to a known nation-state or crime group |

### User risk (is *this account* compromised?)

| Detection | What it means |
|-----------|---------------|
| Leaked credentials | The user's password was found in a credential breach and matched their current hash |
| Microsoft Entra threat intelligence | Activity matching known attack patterns |
| Anomalous token | Odd token lifetime or a token replayed from an unfamiliar location |

### What you can do with risk signals

- Feed them into **Conditional Access** (e.g. "if sign-in risk is high, block"; "if user risk is
  high, force a secure password change")
- Investigate risky users and risky sign-ins in the Identity Protection reports
- Let a user self-remediate: MFA clears sign-in risk, a password reset clears user risk

**Exam tip**: **Entra ID P2** is the answer for ID Protection. A few detections (leaked
credentials, Entra threat intelligence) fire on Free/P1, but without P2 they show only as
*"Additional risk detected"* with no detail, and risk-based Conditional Access is not available.

In [ ]:
# Simulate ID Protection risk detection.
# Detection names match the ones in the Entra ID Protection reports.
SIGN_INS = [
    {'user': 'alice', 'ip': '203.0.113.1',   'location': 'Seattle', 'time': '09:00', 'device': 'laptop-alice',  'detections': []},
    {'user': 'alice', 'ip': '198.51.100.99', 'location': 'Moscow',  'time': '09:30', 'device': 'unknown-device', 'detections': ['atypical_travel', 'unfamiliar_sign_in_properties']},
    {'user': 'bob',   'ip': '10.0.0.50',     'location': 'Office',  'time': '08:45', 'device': 'laptop-bob',    'detections': []},
    {'user': 'carol', 'ip': '192.0.2.1',     'location': 'Tor exit','time': '02:00', 'device': 'unknown',       'detections': ['anonymous_ip_address', 'password_spray']},
]

RISK_SCORES = {
    'atypical_travel': 'high',
    'unfamiliar_sign_in_properties': 'medium',
    'anonymous_ip_address': 'high',
    'password_spray': 'high',
    'malicious_ip_address': 'high',
    'leaked_credentials': 'high',   # user risk, not sign-in risk
}

verdicts = {}
print('=== Identity Protection: Sign-in Risk Analysis ===\n')
for si in SIGN_INS:
    if not si['detections']:
        risk = 'none'
        action = '✅ Allow'
    else:
        risk_levels = [RISK_SCORES.get(d, 'low') for d in si['detections']]
        risk = 'high' if 'high' in risk_levels else 'medium'
        action = '🚫 Block' if risk == 'high' else '🔐 Require MFA'
    verdicts[si['user'], si['location']] = (risk, action)

    print(f'User: {si["user"]}  Location: {si["location"]}  Device: {si["device"]}')
    print(f'  Detections: {si["detections"] or "none"}')
    print(f'  Risk level: {risk}')
    print(f'  Action: {action}\n')

# ID Protection only teaches anything if the clean sign-ins stay clean and the
# dirty ones escalate. Pin both ends.
assert verdicts[('alice', 'Seattle')] == ('none', '✅ Allow'), 'a normal sign-in must not be flagged'
assert verdicts[('bob', 'Office')] == ('none', '✅ Allow')
assert verdicts[('alice', 'Moscow')][0] == 'high', 'atypical travel is a high-risk detection'
assert verdicts[('carol', 'Tor exit')] == ('high', '🚫 Block'), 'Tor + password spray must block'
print('✅ risk scoring separates the clean sign-ins from the risky ones')

---
## 4. Entitlement Management (access packages)

**Problem**: onboarding a new hire or a partner usually means a ticket storm — "add Bob to the Sales SharePoint, the CRM app, the Teams channel, the shared mailbox…".

**Entitlement Management** bundles those resources into an **access package** that users can *request* from a catalog. A policy controls who can request, who approves, and for how long.

### Access package anatomy

| Component | Example |
|-----------|---------|
| **Catalog** | "Sales team resources" |
| **Resources** | Sales SharePoint site, Salesforce app, Sales Teams channel |
| **Policy** | Who can request (internal users / specific partner tenants), who approves, how long access lasts, recurring review |

This is also how you grant **B2B guest** access cleanly: a partner requests the package, an internal owner approves, the guest account is auto-provisioned, and access auto-expires.

**Exam tip**: Entitlement Management is part of **Entra ID Governance** (requires Premium P2 or Entra ID Governance license).

In [ ]:
# Mini simulation of an access package request
ACCESS_PACKAGES = {
    'sales-onboarding': {
        'resources': ['SharePoint: Sales site','App: Salesforce','Group: Sales-All'],
        'approver': 'sales-manager@contoso.com',
        'duration_days': 180,
        'requires_review': True,
    }
}

def request_access_package(user, package, justification, approved):
    pkg = ACCESS_PACKAGES[package]
    if not justification:
        return '❌ request rejected — justification required'
    if not approved:
        return f'⏳ pending approval by {pkg["approver"]}'
    return {
        'user': user,
        'package': package,
        'granted_resources': pkg['resources'],
        'expires_in_days': pkg['duration_days'],
        'review_required': pkg['requires_review'],
        'status': '✅ access granted (auto-expires, recurring review scheduled)',
    }

print('New hire requesting access:')
print(json.dumps(request_access_package('newhire@contoso.com','sales-onboarding','Joining the Sales team Monday', True), indent=2, default=str))

print('\nSame request with no justification:')
print(request_access_package('newhire@contoso.com','sales-onboarding','', True))

granted = request_access_package('newhire@contoso.com', 'sales-onboarding', 'Joining Sales', True)
assert granted['expires_in_days'] == 180, 'access-package assignments must be time-bound'
assert granted['review_required'] is True
assert request_access_package('x@contoso.com', 'sales-onboarding', 'reason', False).startswith('⏳'), \
    'an unapproved request must sit pending, not grant access'
assert request_access_package('x@contoso.com', 'sales-onboarding', '', True).startswith('❌')
print('✅ access packages are approval-gated, time-bound and review-scheduled')

---
## Summary — Identity Governance

| Capability | Purpose | Licence |
|------------|---------|---------|
| **PIM** | Just-in-time admin access | Entra ID P2 **or** Entra ID Governance |
| **Access Reviews** | Periodic verification of access | Entra ID P2 **or** Entra ID Governance |
| **Entitlement Management** | Self-service access packages | Entra ID P2 **or** Entra ID Governance |
| **Lifecycle Workflows** | Automated joiner/mover/leaver tasks | Entra ID Governance only |
| **ID Protection** | ML-based risk detection | Entra ID P2 |

### Exam cheat sheet

- PIM = **time-limited** admin roles, with justification + approval. Eligible ≠ active.
- Access reviews = **periodic** check that access is still needed, with auto-apply on no response.
- Entitlement management = **self-service** access packages that expire on their own.
- ID Protection = **automatic** detection of risky sign-ins and risky users, consumed by
  Conditional Access. It is a risk service, not part of the Governance licence.
- The governance three (PIM, access reviews, entitlement management) came from **P2** and are
  now also sold as **Microsoft Entra ID Governance**; the newer governance features are
  Governance-only.

---
## Self-check — identity governance

Edit `MY_ANSWERS`, re-run, and read the explanations for anything you missed.

In [ ]:
QUIZ = [
    {'id': 'Q1',
     'q': 'An admin is "eligible" for Global Administrator in PIM. What access do they have right now?',
     'options': {'A': 'Full Global Administrator', 'B': 'None of that role until they activate it',
                 'C': 'Read-only Global Administrator', 'D': 'Global Administrator for 4 hours a day'},
     'a': 'B',
     'why': 'Eligible means they *can* activate the role, subject to justification, MFA and possibly '
            'approval. Until then they hold none of its permissions. An *active* assignment is the one '
            'that grants the role now — that is exactly the standing admin PIM exists to eliminate.'},
    {'id': 'Q2',
     'q': 'A user moved from Marketing to Finance two years ago and still has the Marketing SharePoint. '
          'Which capability is designed to catch this?',
     'options': {'A': 'Identity Protection', 'B': 'Conditional Access',
                 'C': 'Access reviews', 'D': 'Password protection'},
     'a': 'C',
     'why': 'This is permission creep, and access reviews are the periodic recertification that finds '
            'it — reviewers approve or deny, and denied access can be removed automatically. '
            'ID Protection scores risky sign-ins; it has no opinion about stale group membership.'},
    {'id': 'Q3',
     'q': 'Which detection is a USER risk (the account looks compromised), not a sign-in risk?',
     'options': {'A': 'Atypical travel', 'B': 'Anonymous IP address',
                 'C': 'Leaked credentials', 'D': 'Unfamiliar sign-in properties'},
     'a': 'C',
     'why': 'Leaked credentials means the password turned up in a breach dump and matched — that is a '
            'property of the account, not of one session. The other three describe a particular '
            'sign-in attempt. Remediation differs too: sign-in risk clears with MFA, user risk with a '
            'secure password change.'},
    {'id': 'Q4',
     'q': 'A partner needs the Sales SharePoint site, the CRM app and a Teams channel, approved by the '
          'Sales manager, expiring after 180 days. What do you build?',
     'options': {'A': 'A Conditional Access policy', 'B': 'An entitlement management access package',
                 'C': 'A PIM eligible assignment', 'D': 'A dynamic group'},
     'a': 'B',
     'why': 'An access package bundles resources with a policy for who may request, who approves, and '
            'how long access lasts — and it can provision the B2B guest and remove it on expiry. PIM is '
            'for privileged roles, not for a bundle of collaboration resources.'},
    {'id': 'Q5',
     'q': 'Which statement about Microsoft Entra ID Protection is correct?',
     'options': {'A': 'It is part of the Microsoft Entra ID Governance licence',
                 'B': 'It blocks risky sign-ins by itself, without Conditional Access',
                 'C': 'It produces risk signals that Conditional Access can act on, and needs P2',
                 'D': 'It replaces MFA'},
     'a': 'C',
     'why': 'ID Protection detects and scores risk; the enforcement decision belongs to Conditional '
            'Access (or to a user-risk password-change policy). It sits in Entra ID P2, separately from '
            'the Entra ID Governance product that covers PIM, access reviews and entitlement management.'},
    {'id': 'Q6',
     'q': 'An access review finishes and several reviewers never responded. What does "auto-apply '
          'results" plus "remove access" do?',
     'options': {'A': 'Nothing — unanswered means unchanged',
                 'B': 'Access for the un-reviewed users is removed automatically',
                 'C': 'The review restarts', 'D': 'The users are deleted from the tenant'},
     'a': 'B',
     'why': 'You choose in advance what a non-response means: approve, no change, or remove. Choosing '
            'remove makes silence fail closed. It removes the *access*, never the user object.'},
]

MY_ANSWERS = {'Q1': 'B', 'Q2': 'C', 'Q3': 'C', 'Q4': 'B', 'Q5': 'C', 'Q6': 'B'}

score = 0
for q in QUIZ:
    mine = MY_ANSWERS.get(q['id'], '').strip().upper()
    ok = mine == q['a']
    score += ok
    print(f'{"PASS" if ok else "FAIL"}  {q["id"]}: {q["q"]}')
    for k, v in q['options'].items():
        mark = '<-- correct' if k == q['a'] else ''
        print(f'         {k}. {v} {mark}')
    print(f'         your answer: {mine or "(blank)"}')
    print(f'         why: {q["why"]}\n')
print(f'Score: {score}/{len(QUIZ)}')

assert {q['id'] for q in QUIZ} == set(MY_ANSWERS), 'every question needs an answer key entry'
assert all(q['a'] in q['options'] for q in QUIZ), 'an answer key points at an option that does not exist'


**Next lab**: [03 — Azure Security Solutions](../../03-azure-security-solutions/)